# 05 — Revision analysis and reviewer-facing tables

Run this **after** Notebooks 01–04.

It creates the analyses requested across Reviewers 1, 2, 3 and 5:

- mean ± SD across three random seeds;
- bootstrap 95% CIs;
- paired bootstrap comparisons for the matched 29-frame conditions;
- phase-weight sensitivity;
- split and score distributions;
- phase/body-part results;
- stroke-type subgroup performance;
- pose-confidence/observability by phase;
- calibration and residual diagnostics;
- Bland–Altman agreement;
- manuscript-ready CSV tables and figures.

The notebook does not retrain any models.

In [ ]:
from pathlib import Path
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, pearsonr

REVISION_ROOT = Path(os.environ.get("AQA_REVISION_ROOT", str(Path.cwd() / "artifacts"))).expanduser().resolve()
CACHE_DIR = REVISION_ROOT / "cache"
RESULTS_DIR = REVISION_ROOT / "results"
ANALYSIS_DIR = REVISION_ROOT / "analysis"
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

MODELS = ["RGB12_GROUPED", "RGB29_GROUPED", "POSE29_GROUPED", "RGBPOSE29_GROUPED"]
SEEDS = [42, 123, 2026]
PHASE_NAMES = ["Buildup", "Execution", "FollowThrough"]
BODY_PARTS = ["Head", "Shoulders", "Hands", "Hips", "Feet"]
DEFAULT_W = np.asarray([0.25, 0.50, 0.25], dtype=np.float32)

manifest = pd.read_csv(CACHE_DIR / "manifest_with_split.csv")
y_all = np.load(CACHE_DIR / "y_scores.npy")
test_idx = np.load(CACHE_DIR / "test_idx.npy")
val_idx = np.load(CACHE_DIR / "val_idx.npy")
train_idx = np.load(CACHE_DIR / "train_idx.npy")

def safe_spearman(a, b):
    if np.unique(a).size < 2 or np.unique(b).size < 2:
        return np.nan
    return float(spearmanr(a, b).correlation)

def safe_pearson(a, b):
    if np.unique(a).size < 2 or np.unique(b).size < 2:
        return np.nan
    return float(pearsonr(a, b)[0])

def phase_and_overall(y, weights=DEFAULT_W):
    phase = np.asarray(y).mean(axis=2)
    overall = np.sum(phase * np.asarray(weights)[None, :], axis=1)
    return phase, overall

def metric_triplet(y_true, y_pred, weights=DEFAULT_W):
    _, t = phase_and_overall(y_true, weights)
    _, p = phase_and_overall(y_pred, weights)
    return {
        "SRC": safe_spearman(t, p),
        "Pearson": safe_pearson(t, p),
        "MAE": float(np.mean(np.abs(t-p))),
        "RMSE": float(np.sqrt(np.mean((t-p)**2))),
    }

def load_predictions(model, seed):
    p = RESULTS_DIR / model / f"seed_{seed}" / "predictions.npz"
    z = np.load(p)
    return {k: z[k] for k in z.files}

In [ ]:
# ============================================================
# 1. Repeated-run variability: mean ± SD across seeds
# ============================================================
rows = []
for model in MODELS:
    model_csv = RESULTS_DIR / model / "all_seeds_metrics.csv"
    if not model_csv.exists():
        raise FileNotFoundError(f"Missing {model_csv}. Run its training notebook first.")
    df = pd.read_csv(model_csv)
    for _, r in df.iterrows():
        rows.append(r.to_dict())

runs_df = pd.DataFrame(rows)
runs_df.to_csv(ANALYSIS_DIR / "all_model_seed_metrics.csv", index=False)

cols = [
    "test_overall_SRC", "test_overall_Pearson", "test_overall_MAE", "test_overall_RMSE",
    "test_Buildup_SRC", "test_Execution_SRC", "test_FollowThrough_SRC"
]
summary_rows = []
for model, g in runs_df.groupby("model"):
    row = {"model": model, "n_seeds": len(g)}
    for c in cols:
        row[c + "_mean"] = g[c].mean()
        row[c + "_sd"] = g[c].std(ddof=1)
    summary_rows.append(row)

seed_summary = pd.DataFrame(summary_rows)
display(seed_summary)
seed_summary.to_csv(ANALYSIS_DIR / "seed_variability_summary.csv", index=False)

In [ ]:
# ============================================================
# 2. Bootstrap CIs and paired bootstrap comparisons
# Primary paired comparisons use seed 42; the three seeds quantify run variability.
# ============================================================
def bootstrap_ci(y_true, y_pred, metric_fn, n_boot=5000, seed=42):
    rng = np.random.RandomState(seed)
    n = len(y_true)
    vals = np.empty(n_boot, dtype=float)
    for b in range(n_boot):
        idx = rng.randint(0, n, n)
        vals[b] = metric_fn(y_true[idx], y_pred[idx])
    return np.nanpercentile(vals, [2.5, 97.5])

def paired_bootstrap_delta(y_true, pred_a, pred_b, metric_fn, higher_better=True, n_boot=5000, seed=42):
    """
    Positive delta favours model B.
    For higher-is-better metrics: B - A.
    For MAE: A_MAE - B_MAE.
    """
    rng = np.random.RandomState(seed)
    n = len(y_true)
    vals = np.empty(n_boot, dtype=float)
    for b in range(n_boot):
        idx = rng.randint(0, n, n)
        ma = metric_fn(y_true[idx], pred_a[idx])
        mb = metric_fn(y_true[idx], pred_b[idx])
        vals[b] = (mb - ma) if higher_better else (ma - mb)
    return float(np.nanmean(vals)), np.nanpercentile(vals, [2.5, 97.5])

def overall_arrays(pred_pack):
    y_true = pred_pack["y_test"]
    y_pred = pred_pack["pred_test"]
    _, t = phase_and_overall(y_true)
    _, p = phase_and_overall(y_pred)
    return t, p

ci_rows = []
for model in MODELS:
    z = load_predictions(model, 42)
    t, p = overall_arrays(z)
    for metric_name, fn in [
        ("SRC", safe_spearman),
        ("Pearson", safe_pearson),
        ("MAE", lambda a,b: float(np.mean(np.abs(a-b)))),
    ]:
        est = fn(t, p)
        lo, hi = bootstrap_ci(t, p, fn, n_boot=5000, seed=42)
        ci_rows.append({
            "model": model, "seed": 42, "metric": metric_name,
            "estimate": est, "ci_low": lo, "ci_high": hi
        })

ci_df = pd.DataFrame(ci_rows)
display(ci_df)
ci_df.to_csv(ANALYSIS_DIR / "bootstrap_95ci_seed42.csv", index=False)

pairs = [
    ("RGB29_GROUPED", "RGBPOSE29_GROUPED", "RGB+Pose-29 over RGB-29"),
    ("RGBPOSE29_GROUPED", "POSE29_GROUPED", "Pose-29 over RGB+Pose-29"),
]
comparisons = []
for model_a, model_b, label in pairs:
    a = load_predictions(model_a, 42)
    b = load_predictions(model_b, 42)
    t_a, p_a = overall_arrays(a)
    t_b, p_b = overall_arrays(b)
    assert np.allclose(t_a, t_b)

    for metric_name, fn, higher in [
        ("SRC", safe_spearman, True),
        ("Pearson", safe_pearson, True),
        ("MAE", lambda x,y: float(np.mean(np.abs(x-y))), False),
    ]:
        delta, ci = paired_bootstrap_delta(
            t_a, p_a, p_b, fn, higher_better=higher, n_boot=5000, seed=42
        )
        comparisons.append({
            "comparison": label,
            "metric": metric_name,
            "delta_favouring_second_model": delta,
            "ci_low": ci[0],
            "ci_high": ci[1],
        })

paired_df = pd.DataFrame(comparisons)
display(paired_df)
paired_df.to_csv(ANALYSIS_DIR / "paired_bootstrap_model_differences_seed42.csv", index=False)


In [ ]:
# ============================================================
# 3. Temporal sampling comparison: RGB-12 vs RGB-29
# ============================================================
temporal_rows = []
for seed in SEEDS:
    a = load_predictions("RGB12_GROUPED", seed)
    b = load_predictions("RGB29_GROUPED", seed)

    ma = metric_triplet(a["y_test"], a["pred_test"])
    mb = metric_triplet(b["y_test"], b["pred_test"])

    temporal_rows.append({
        "seed": seed,
        "RGB12_SRC": ma["SRC"], "RGB29_SRC": mb["SRC"],
        "Delta_SRC_29_minus_12": mb["SRC"] - ma["SRC"],
        "RGB12_MAE": ma["MAE"], "RGB29_MAE": mb["MAE"],
        "Delta_MAE_12_minus_29": ma["MAE"] - mb["MAE"],
    })

temporal_df = pd.DataFrame(temporal_rows)
display(temporal_df)
temporal_df.to_csv(ANALYSIS_DIR / "rgb12_vs_rgb29_temporal_ablation.csv", index=False)

In [ ]:
# ============================================================
# 4. Phase-weight sensitivity — NO optimisation on test data
# ============================================================
WEIGHT_SCHEMES = {
    "Original_025_050_025": np.asarray([0.25, 0.50, 0.25]),
    "Equal_thirds": np.asarray([1/3, 1/3, 1/3]),
    "Execution_heavy_020_060_020": np.asarray([0.20, 0.60, 0.20]),
}

weight_rows = []
for model in ["RGB29_GROUPED", "POSE29_GROUPED", "RGBPOSE29_GROUPED"]:
    for seed in SEEDS:
        z = load_predictions(model, seed)
        for name, w in WEIGHT_SCHEMES.items():
            m = metric_triplet(z["y_test"], z["pred_test"], weights=w)
            weight_rows.append({
                "model": model, "seed": seed, "weight_scheme": name, **m
            })

weight_df = pd.DataFrame(weight_rows)
display(weight_df)
weight_df.to_csv(ANALYSIS_DIR / "phase_weight_sensitivity.csv", index=False)

In [ ]:
# ============================================================
# 5. Ground-truth score distributions and split characteristics
# ============================================================
def describe(v):
    v = np.asarray(v, dtype=float)
    return {
        "N": len(v),
        "Mean": np.mean(v),
        "SD": np.std(v, ddof=1),
        "Median": np.median(v),
        "IQR": np.percentile(v, 75) - np.percentile(v, 25),
        "Min": np.min(v),
        "Max": np.max(v),
    }

split_rows = []
phase_rows = []

for split_name, idx in [("Train", train_idx), ("Validation", val_idx), ("Test", test_idx)]:
    phase, overall = phase_and_overall(y_all[idx])
    row = {
        "Split": split_name,
        "Source_videos": manifest.iloc[idx]["source_video_id"].nunique(),
        **describe(overall),
    }
    split_rows.append(row)

    for p, name in enumerate(PHASE_NAMES):
        phase_rows.append({"Split": split_name, "Phase": name, **describe(phase[:, p])})

split_dist_df = pd.DataFrame(split_rows)
phase_dist_df = pd.DataFrame(phase_rows)

display(split_dist_df)
display(phase_dist_df)

split_dist_df.to_csv(ANALYSIS_DIR / "split_overall_score_distribution.csv", index=False)
phase_dist_df.to_csv(ANALYSIS_DIR / "split_phase_score_distribution.csv", index=False)

# Stroke-type counts by split
stroke_counts = (
    manifest.groupby(["split", "Stroke_Type"], dropna=False)
    .size().reset_index(name="N")
    .sort_values(["split", "N"], ascending=[True, False])
)
stroke_counts.to_csv(ANALYSIS_DIR / "stroke_type_counts_by_split.csv", index=False)
display(stroke_counts.head(30))

In [ ]:
# ============================================================
# 6. Phase/body-part results for the matched Pose-29 model (seed 42)
# ============================================================
z = load_predictions("POSE29_GROUPED", 42)
yt, yp = z["y_test"], z["pred_test"]

pb_rows = []
for p, phase in enumerate(PHASE_NAMES):
    for b, body in enumerate(BODY_PARTS):
        t = yt[:, p, b]
        q = yp[:, p, b]
        pb_rows.append({
            "Phase": phase,
            "Body_part": body,
            "SRC": safe_spearman(t, q),
            "Pearson": safe_pearson(t, q),
            "MAE": float(np.mean(np.abs(t-q))),
        })

phase_body_df = pd.DataFrame(pb_rows)
display(phase_body_df)
phase_body_df.to_csv(ANALYSIS_DIR / "pose29_seed42_phase_by_bodypart.csv", index=False)

In [ ]:
# ============================================================
# 7. Stroke-type subgroup analysis on held-out test set
# Only groups with N >= 30 are reported.
# ============================================================
test_meta = manifest.iloc[test_idx].reset_index(drop=True)
z = load_predictions("POSE29_GROUPED", 42)
true_phase, true_overall = phase_and_overall(z["y_test"])
pred_phase, pred_overall = phase_and_overall(z["pred_test"])

sub_rows = []
for stroke_type, idx_local in test_meta.groupby("Stroke_Type").groups.items():
    idx_local = np.asarray(list(idx_local), dtype=int)
    if len(idx_local) < 30:
        continue
    t = true_overall[idx_local]
    p = pred_overall[idx_local]
    sub_rows.append({
        "Stroke_Type": stroke_type,
        "N": len(idx_local),
        "SRC": safe_spearman(t, p),
        "Pearson": safe_pearson(t, p),
        "MAE": float(np.mean(np.abs(t-p))),
        "True_mean": float(np.mean(t)),
        "Pred_mean": float(np.mean(p)),
    })

stroke_perf_df = pd.DataFrame(sub_rows).sort_values("N", ascending=False)
display(stroke_perf_df)
stroke_perf_df.to_csv(ANALYSIS_DIR / "pose29_test_stroke_type_performance_n_ge_30.csv", index=False)

In [ ]:
# ============================================================
# 8. Pose quality by phase
# ============================================================
pose_conf = np.load(CACHE_DIR / "pose29_confidence.npy", mmap_mode="r")
pose_obs = np.load(CACHE_DIR / "pose29_observed.npy", mmap_mode="r")

phase_slices = [(0,14), (14,21), (21,29)]
pose_rows = []

for split_name, idx in [("Train", train_idx), ("Validation", val_idx), ("Test", test_idx)]:
    for (s,e), phase in zip(phase_slices, PHASE_NAMES):
        conf = np.asarray(pose_conf[idx, s:e, :], dtype=np.float32).ravel()
        obs = np.asarray(pose_obs[idx, s:e, :], dtype=np.float32).ravel()

        pose_rows.append({
            "Split": split_name,
            "Phase": phase,
            "Mean_keypoint_confidence": float(np.mean(conf)),
            "Median_keypoint_confidence": float(np.median(conf)),
            "Observed_keypoint_fraction": float(np.mean(obs)),
            "Low_confidence_fraction_lt_0.05": float(np.mean(conf < 0.05)),
        })

pose_quality_df = pd.DataFrame(pose_rows)
display(pose_quality_df)
pose_quality_df.to_csv(ANALYSIS_DIR / "pose_quality_by_phase.csv", index=False)

In [ ]:
# ============================================================
# 9. Calibration, residuals, score bins and Bland–Altman
# ============================================================
z = load_predictions("POSE29_GROUPED", 42)
_, y_true = phase_and_overall(z["y_test"])
_, y_pred = phase_and_overall(z["pred_test"])

resid = y_pred - y_true
bias = float(np.mean(resid))
sd = float(np.std(resid, ddof=1))
rmse = float(np.sqrt(np.mean(resid**2)))
mae = float(np.mean(np.abs(resid)))
loa_low = bias - 1.96 * sd
loa_high = bias + 1.96 * sd

# Continuous calibration: observed = intercept + slope * predicted
slope, intercept = np.polyfit(y_pred, y_true, 1)

diag = pd.DataFrame([{
    "Bias_pred_minus_true": bias,
    "Residual_SD": sd,
    "MAE": mae,
    "RMSE": rmse,
    "Bland_Altman_lower": loa_low,
    "Bland_Altman_upper": loa_high,
    "Calibration_intercept_observed_on_predicted": intercept,
    "Calibration_slope_observed_on_predicted": slope,
    "SRC": safe_spearman(y_true, y_pred),
    "Pearson": safe_pearson(y_true, y_pred),
}])
display(diag)
diag.to_csv(ANALYSIS_DIR / "pose29_seed42_calibration_and_agreement.csv", index=False)

# Quartile-based score-bin analysis, preserving the existing interpretation.
edges = np.quantile(y_true, [0, 0.25, 0.50, 0.75, 1.0])
bin_rows = []
for i in range(4):
    if i < 3:
        m = (y_true >= edges[i]) & (y_true < edges[i+1])
    else:
        m = (y_true >= edges[i]) & (y_true <= edges[i+1])

    t, p = y_true[m], y_pred[m]
    bin_rows.append({
        "Bin": i+1,
        "Lower": edges[i],
        "Upper": edges[i+1],
        "N": len(t),
        "Mean_true": np.mean(t),
        "Mean_pred": np.mean(p),
        "Bias_pred_minus_true": np.mean(p-t),
        "MAE": np.mean(np.abs(p-t)),
        "RMSE": np.sqrt(np.mean((p-t)**2)),
    })

bin_df = pd.DataFrame(bin_rows)
display(bin_df)
bin_df.to_csv(ANALYSIS_DIR / "pose29_seed42_score_bin_errors.csv", index=False)

# Prediction + residual figure
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(y_true, y_pred, alpha=0.45)
lo = min(y_true.min(), y_pred.min())
hi = max(y_true.max(), y_pred.max())
axes[0].plot([lo, hi], [lo, hi], linestyle="--")
axes[0].set_xlabel("Ground-truth overall score")
axes[0].set_ylabel("Predicted overall score")
axes[0].set_title(
    f"Test prediction agreement\nSRC={safe_spearman(y_true,y_pred):.3f}, "
    f"Pearson={safe_pearson(y_true,y_pred):.3f}, MAE={mae:.3f}"
)

axes[1].scatter(y_true, resid, alpha=0.45)
axes[1].axhline(0, linestyle="--")
axes[1].set_xlabel("Ground-truth overall score")
axes[1].set_ylabel("Residual (predicted - true)")
axes[1].set_title("Test residuals across score range")

plt.tight_layout()
plt.savefig(ANALYSIS_DIR / "pose29_prediction_and_residuals.png", dpi=300, bbox_inches="tight")
plt.show()

# Bland–Altman
means = (y_true + y_pred) / 2
plt.figure(figsize=(7,5))
plt.scatter(means, resid, alpha=0.45)
plt.axhline(bias, linestyle="--", label=f"Bias={bias:.3f}")
plt.axhline(loa_low, linestyle="--", label=f"Lower LoA={loa_low:.3f}")
plt.axhline(loa_high, linestyle="--", label=f"Upper LoA={loa_high:.3f}")
plt.xlabel("Mean of ground-truth and predicted score")
plt.ylabel("Predicted - ground truth")
plt.title("Bland–Altman analysis")
plt.legend()
plt.tight_layout()
plt.savefig(ANALYSIS_DIR / "pose29_bland_altman.png", dpi=300, bbox_inches="tight")
plt.show()

# Calibration plot
plt.figure(figsize=(6,5))
plt.scatter(y_pred, y_true, alpha=0.45)
xx = np.linspace(y_pred.min(), y_pred.max(), 100)
plt.plot(xx, xx, linestyle="--", label="Ideal y=x")
plt.plot(xx, intercept + slope*xx, label=f"Observed={intercept:.2f}+{slope:.2f}×Predicted")
plt.xlabel("Predicted overall score")
plt.ylabel("Observed overall score")
plt.title("Continuous calibration")
plt.legend()
plt.tight_layout()
plt.savefig(ANALYSIS_DIR / "pose29_calibration_plot.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# ============================================================
# 10. Final reviewer-facing matched-ablation table
# ============================================================
rows = []
for model in MODELS:
    for seed in SEEDS:
        z = load_predictions(model, seed)
        mm = metric_triplet(z["y_test"], z["pred_test"])
        rows.append({"Model": model, "Seed": seed, **mm})

ablation_runs = pd.DataFrame(rows)

final_rows = []
for model, g in ablation_runs.groupby("Model"):
    final_rows.append({
        "Model": model,
        "SRC_mean": g["SRC"].mean(),
        "SRC_SD": g["SRC"].std(ddof=1),
        "Pearson_mean": g["Pearson"].mean(),
        "Pearson_SD": g["Pearson"].std(ddof=1),
        "MAE_mean": g["MAE"].mean(),
        "MAE_SD": g["MAE"].std(ddof=1),
        "RMSE_mean": g["RMSE"].mean(),
        "RMSE_SD": g["RMSE"].std(ddof=1),
    })

final_ablation = pd.DataFrame(final_rows)
display(final_ablation)
final_ablation.to_csv(ANALYSIS_DIR / "FINAL_MATCHED_ABLATION_TABLE.csv", index=False)

print("\nAnalysis complete.")
print("Reviewer-facing outputs are in:", ANALYSIS_DIR)